# Jacobian lens — walkthrough

Load a model, load a pre-fitted Jacobian lens from the Hub, apply it to a prompt, and render the interactive slice visualisation.

In [2]:
! cd /content && git clone https://github.com/taot/jacobian-lens.git

Cloning into 'jacobian-lens'...
remote: Enumerating objects: 67, done.
remote: Counting objects: 100% (3/3), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 67 (delta 0), reused 0 (delta 0), pack-reused 64 (from 1)
Receiving objects: 100% (67/67), 1.74 MiB | 4.40 MiB/s, done.
Resolving deltas: 100% (8/8), done.


In [9]:
%pip install /content/jacobian-lens

Processing ./jacobian-lens
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for jlens: filename=jlens-0.1.0-py3-none-any.whl size=49953 sha256=65f6333ae255cd696fc0cb3254b822d00137817408969b5435c43d0e768aaaf2
  Stored in directory: /root/.cache/pip/wheels/9b/16/f6/ff5117e12d375117559a1ded186e4d458b172d145efd7f032b
Successfully built jlens
  Attempting uninstall: jlens
    Found existing installation: jlens 0.1.0
    Uninstalling jlens-0.1.0:
      Successfully uninstalled jlens-0.1.0


In [10]:
import jlens

jlens.configure_logging()

MODEL_NAME = "Qwen/Qwen3.5-4B"
# MODEL_NAME = "Qwen/Qwen3.6-27B"

LENS_REPO = "neuronpedia/jacobian-lens"
LENS_REVISION = "qwen-n1000"
LENS_FILE = {
    "Qwen/Qwen3.5-4B": "qwen3.5-4b/jlens/Salesforce-wikitext/Qwen3.5-4B_jacobian_lens_n1000.pt",
    "Qwen/Qwen3.6-27B": "qwen3.6-27b/jlens/Salesforce-wikitext/Qwen3.6-27B_jacobian_lens_n1000.pt",
}[MODEL_NAME]

## 1. Load the model

`jlens.from_hf` wraps an already-loaded HuggingFace model into `LensModel` interface

In [11]:
import torch
import transformers

hf_model = transformers.AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, dtype=torch.bfloat16
).cuda()
tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL_NAME)
model = jlens.from_hf(hf_model, tokenizer)
model

config.json:   0%|          | 0.00/3.16k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/76.2k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

...

merges.txt:   0%|          | 0.00/3.35M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 12.8MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/7.76k [00:00<?, ?B/s]

HFLensModel(Qwen3_5ForCausalLM, n_layers=32, d_model=2560)

## 2. Load a pre-fitted lens

`JacobianLens.from_pretrained` pulls a `.pt` from the Hub (or a local path). The lens holds one `[d_model, d_model]` matrix per layer.

In [12]:
lens = jlens.JacobianLens.from_pretrained(
    LENS_REPO, filename=LENS_FILE, revision=LENS_REVISION
)
lens

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

JacobianLens(d_model=2560, n_prompts=1000, source_layers=[0..30] (31 layers))

## 3. Apply: J-lens vs logit lens

`lens.apply(model, prompt, positions=...)` runs one forward pass, transports each layer's residual into the final-layer basis with `J_l`, and decodes through the model's own unembedding. `use_jacobian=False` skips the transport — that's the vanilla logit lens.

Below: a two-hop factual question, read out at the boot token. The J-lens surfaces interpretable tokens at layers where the logit lens is still noise.

In [13]:
prompt = "Fact: The currency used in the country shaped like a boot is"
layers = [
    model.n_layers // 4,
    model.n_layers // 2,
    model.n_layers // 4 * 3,
    model.n_layers - 2,
]

jlens_logits, model_logits, _ = lens.apply(model, prompt, layers=layers, positions=[-2])
logit_lens, _, _ = lens.apply(
    model, prompt, layers=layers, positions=[-2], use_jacobian=False
)


def top5(logits):
    return [tokenizer.decode([t]) for t in logits.topk(5).indices]


for layer in layers:
    print(f"L{layer:>3} logit-lens: {top5(logit_lens[layer][0])}")
    print(f"L{layer:>3} J-lens:     {top5(jlens_logits[layer][0])}")
print(f"model:           {top5(model_logits[0])}")

L  8 logit-lens: ['oman', 'edom', 'ולי', ' Urlaubs', 'GPC']
L  8 J-lens:     [' `', ' boots', ' *', ' `\\', ' heel']
L 16 logit-lens: ['shaw', 'วย', 'amaz', 'REA', '举世']
L 16 J-lens:     ['?', '？', "'?", ' Italy', '____']
L 24 logit-lens: ['的形状', '形状的', 'shape', '形状', '-shaped']
L 24 J-lens:     ['-shaped', ' shape', ' shaped', 'shape', '形状']
L 30 logit-lens: [' is', ' shape', '-shaped', ' heel', ' shaped']
L 30 J-lens:     [' is', ' shape', ' shaped', '-shaped', ' heel']
model:           [' is', ' in', ' on', '.', ' with']


## 4. Render a slice page (inline)

`compute_slice` + `build_page` produce an interactive position × layer view of the lens's token ranks (the `?` in the corner explains the controls). `mode="embed"` inlines everything so the page is self-contained.

In [16]:
import gzip
import json

from jlens.examples import EXAMPLES, resolve_prompt
from jlens.vis import build_page, compute_slice, notebook_iframe

# English gloss for Qwen's Chinese/Japanese/Korean vocab tokens (machine-
# generated, best-effort), shown next to
# the token in the page (alt_token=).
gloss = {
    int(k): v for k, v in json.load(gzip.open("/content/jacobian-lens/assets/qwen_gloss.json.gz")).items()
}

example = next(e for e in EXAMPLES if e.slug == "multihop")
prompt = resolve_prompt(example, tokenizer)

slice_data = compute_slice(
    model,
    lens,
    prompt,
    layer_stride=2,
    # Empirically on Qwen, the interesting word tokens trail punctuation and
    # single-character tokens in the raw top-K; mask to word-like tokens only.
    mask_display=True,
)
page, _, _ = build_page(
    slice_data,
    prompt,
    title=example.section,
    description=example.description,
    alt_token=gloss,
)
notebook_iframe(page)

## 5. Render a slice page (served)

For longer prompts prefer `mode="fetch"`: `build_page` writes the data as sidecar files to `out_dir` and the page fetches rank files lazily on pin, so it stays small regardless of how many tokens are tracked.

In [22]:
import os
import threading
from functools import partial
from http.server import HTTPServer, SimpleHTTPRequestHandler
from pathlib import Path

example = next(e for e in EXAMPLES if e.slug == "multihop") # ascii-face
prompt = resolve_prompt(example, tokenizer)

slice_data = compute_slice(model, lens, prompt, mask_display=True)
out_dir = Path("slices") / example.slug
page, _, _ = build_page(
    slice_data,
    prompt,
    title=example.section,
    description=example.description,
    alt_token=gloss,
    mode="fetch",
    out_dir=out_dir,
)
(out_dir / "index.html").write_text(page)

if "_jlens_httpd" not in globals():
    _handler = partial(SimpleHTTPRequestHandler, directory=os.path.abspath("slices"))
    _jlens_httpd = HTTPServer(("127.0.0.1", 0), _handler)
    threading.Thread(target=_jlens_httpd.serve_forever, daemon=True).start()
print(f"-> http://localhost:{_jlens_httpd.server_address[1]}/{example.slug}/")

-> http://localhost:33795/multihop/


### Expose through pyngrok

In [23]:
%pip install pyngrok

In [24]:
from pyngrok import ngrok

# Used to securely store your API key
from google.colab import userdata

# Get ngrok auth token from Colab Secrets
NGROK_AUTH_TOKEN=userdata.get('NGROK_AUTH_TOKEN')

# Authenticate ngrok
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# Open a tunnel to the local server port
public_url = ngrok.connect(_jlens_httpd.server_address[1])

# Print the public URL
print(f"Ngrok Public URL: {public_url}/{example.slug}/")

Ngrok Public URL: NgrokTunnel: "https://snuggle-mardi-scolding.ngrok-free.dev" -> "http://localhost:33795"/multihop/


In [25]:
from pyngrok import ngrok

# Disconnect all active ngrok tunnels and terminate the ngrok process
ngrok.kill()
print("pyngrok tunnels and processes have been stopped.")

pyngrok tunnels and processes have been stopped.


### My exploration

In [20]:
# Using the transformers library's built-in generate method to avoid repetition
input_ids = tokenizer.encode(prompt, return_tensors='pt').cuda()

# Generate with repetition penalty and sampling to improve variety
output_ids = model._hf_model.generate(
    input_ids,
    max_length=1200,
    repetition_penalty=1.2,
    do_sample=True,
    top_p=0.95,
    temperature=0.7,
    pad_token_id=tokenizer.eos_token_id
)

generated_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)

print(f"Prompt: {prompt}")
print("\nGenerated text:")
print(generated_text)

Prompt:      _______     
   /         \   
  /  ~     ~  \  
 (   o     o   ) 
 |      ^      | 
 |             | 
 |   \_____/   | 
  \           /  
   \_________/   
      |   |      

What is this?

Generated text:
     _______     
   /         \   
  /  ~     ~  \  
 (   o     o   ) 
 |      ^      | 
 |             | 
 |   \_____/   | 
  \           /  
   \_________/   
      |   |      

What is this?

<think>
Thinking Process:

1.  **Analyze the Request:** The user has provided an ASCII art image and asked "What is this?". I need to identify what the drawing represents or if it's a known meme/template.

2.  **Examine the Image Content:**
    *   It looks like text drawn using characters on lines of code/monospace font style.
    *   Structure: Top line `_______` (looks like roof), second `\` `/`, third `~` `~`. Fourth `(o)`, fifth `| |`, sixth empty, seventh `__\/__/`, eighth `\`, ninth `_`... wait let me look closer at the structure as a whole shape.
    *   Shape analysis:

### More to explore

A few more prompts are bundled in `jlens.examples.EXAMPLES` — change the `slug` above and see what surfaces, or try a prompt of your own.

In [ ]:
for e in EXAMPLES:
    print(f"{e.slug:>24}  {e.section}")

## 6. Fitting

`fit(model, prompts)` computes `J_l` over the supplied prompts. 100 prompts is enough for a usable lens; the released lenses use 1000. `dim_batch` is the memory knob — each prompt does `ceil(d_model / dim_batch)` backward passes on a retained graph.

In [ ]:
from jlens.examples import load_wikitext_prompts

prompts = load_wikitext_prompts(n_prompts=100)
lens = jlens.fit(
    model, prompts, dim_batch=32, max_seq_len=128, checkpoint_path="ckpt.pt"
)
lens.save("jacobian_lens.pt")